# Hướng dẫn chạy HR-VITON Lightweight
___

## 1. Chuẩn bị môi trường
### 1.1. Clone repository

In [10]:
!git clone https://github.com/nnhan727/HR-VITON.git
!git checkout -b Lightweight-Ver
!git status

fatal: destination path 'HR-VITON' already exists and is not an empty directory.
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


### 1.2 Cài thư viện

In [ ]:
!pip install opencv-python pillow tqdm
!pip install tensorboardX
!pip install torchgeometry
!pip install scikit-image
!pip install matplotlib
!pip install lpips
!pip install gdown

___

## 2. Chuẩn bị dataset

**Cách 1: Thêm kaggle dataset vào Input: VITON-HD**
- Cấu trúc dataset như sau:
```
dataset/
├── train/
├── test/
├── image/
├── image-parse/
├── cloth/
├── cloth-mask/
├── openpose_img/
├── openpose_json/
```
- Cho nó về thư mục data/zalando-hd-resized:
```
%mkdir -p data
 %mv dataset data/zalando-hd-resized
```


**Cách 2: Tải lên từ ggdrive**

Tạo các folder trong kaggle working:

In [ ]:
import sys
import os
import gdown

# Define the path to the cloned repo
repo_path = '/kaggle/working/HR-VITON'

# Add it to the system path
if repo_path not in sys.path:
    sys.path.append(repo_path)

os.makedirs('/kaggle/working/HR-VITON/eval_models/weights/v0.1', exist_ok=True)
os.makedirs('/kaggle/working/HR-VITON/data', exist_ok=True)
os.makedirs('/kaggle/working/HR-VITON/results', exist_ok=True)
os.makedirs('/kaggle/working/HR-VITON/checkpoints', exist_ok=True)

# Define files: { "File_ID": "Destination_Path" }
files_to_download = {
    '1h-PZpMthIbycDYCVW6OFxN8jcs5yVNZ9': '/kaggle/working/HR-VITON/data/zalando-hd-resized.zip'
}

for fid, path in files_to_download.items():
    if not os.path.exists(path):
        gdown.download(id=fid, output=path, quiet=False)

In [ ]:
!unzip -q /kaggle/working/HR-VITON/data/zalando-hd-resized.zip -d /kaggle/working/HR-VITON/data/
!rm /kaggle/working/HR-VITON/data/zalando-hd-resized.zip

___

## 3. Train
### 3.1 Train condition-generator

In [ ]:
!python /kaggle/working/HR-VITON/train_condition.py \
  --name condition \
  --gpu_ids 0 \
  -b 4 \
  --dataroot /kaggle/working/HR-VITON/data/zalando-hd-resized

### 3.2. Train Generator (SPADE)

In [ ]:
!python /kaggle/working/HR-VITON/train_generator.py \
  --name generator \
  --gpu_ids 0 \
  -b 4 \
  --dataroot /kaggle/working/HR-VITON/data/zalando-hd-resized

___

## 4. Test (try-on)

In [ ]:
!python /kaggle/working/HR-VITON/test_generator.py \
  --name test \
  --gpu_ids 0 \
  --dataroot /kaggle/working/HR-VITON/data/zalando-hd-resized \
  --checkpoint_dir /kaggle/working/HR-VITON/checkpoints

Output: `results/`

### Visualize

In [ ]:
import glob
import random
import matplotlib.pyplot as plt
from PIL import Image
import os

img_paths = glob.glob('/kaggle/working/HR-VITON/results/*.png')
random_tryon_img_path = random.choice(img_paths)

filename = os.path.basename(random_tryon_img_path)
filename_without_ext = os.path.splitext(filename)[0]

# filename format is 'personID_XX_clothID_YY.png'
parts = filename_without_ext.split('_')
person_id = f'{parts[0]}_{parts[1]}'
cloth_id = f'{parts[2]}_{parts[3]}'

person_img_path = f'/kaggle/working/HR-VITON/data/test/image/{person_id}.jpg'
cloth_img_path = f'/kaggle/working/HR-VITON/data/test/cloth/{cloth_id}.jpg'

person_img = Image.open(person_img_path)
cloth_img = Image.open(cloth_img_path)
tryon_img = Image.open(random_tryon_img_path)

plt.figure(figsize=(10, 4))

plt.subplot(1, 3, 1)
plt.imshow(person_img)
plt.xticks([])
plt.yticks([])
plt.title(f'Person ({person_id})')

plt.subplot(1, 3, 2)
plt.imshow(cloth_img)
plt.xticks([])
plt.yticks([])
plt.title(f'Clothes ({cloth_id})')

plt.subplot(1, 3, 3)
plt.imshow(tryon_img)
plt.xticks([])
plt.yticks([])
plt.title('Try-on')

plt.show()

Chạy file evaluate.py của tác giả để tính toán các chỉ số đánh giá. Các tham số predict_dir và ground_truth_dir được thiết lập để trỏ đến thư mục chứa ảnh kết quả và ảnh mẫu người ban đầu.

In [ ]:
!python /kaggle/working/HR-VITON/evaluate.py \
    --predict_dir '/kaggle/working/HR-VITON/results/' \
    --ground_truth_dir '/kaggle/working/HR-VITON/data/test/image' \
    --resolution 1024